# Bed-Net Use Behaviour Classification from Accelerometer Data
## Complete analysis

Classifying how insecticide-treated bed nets are actually used, from a single accelerometer fixed to the net.

This notebook is the full analysis in one document: data understanding, quality control, exploratory analysis, feature selection, validation design, model comparison, error analysis and interpretation. It is the merged form of the seven chapter notebooks in this repository.

**Runtime:** roughly 5–8 minutes. The model sweeps in Parts 5 and 6 and the permutation importance in Part 7 are the slow cells.

**A note on results:** this notebook publishes the pipeline and methodology, not the findings. The tagged accelerometer extract is human-subjects research data belonging to the study investigators, and specific results (accuracy, class-level performance, confusion patterns, feature rankings) are withheld from this notebook pending the PI's review and sign-off on publication.

---

### Contents

1. [Data and study understanding](#part-1)
2. [Data quality](#part-2)
3. [Exploratory analysis](#part-3)
4. [Feature engineering and selection](#part-4)
5. [Validation design and baseline models](#part-5)
6. [Model comparison and error analysis](#part-6)
7. [Interpretation](#part-7)
8. [Summary and limitations](#part-8)

---

### Scientific context

The methodology follows Koudou et al. (2022), *Evaluation of an accelerometer-based monitor for detecting bed net use and human entry/exit using a machine learning algorithm*, Malaria Journal 21:85. That study established the accelerometer-based classification approach this pipeline follows.

This analysis uses a separate, later tagged dataset. It is not a re-analysis of the published data.

### Data governance

The tagged extract is human-subjects research data belonging to the study investigators. It is excluded from version control, and this analysis's specific findings are withheld pending PI review. This notebook runs on synthetic data of identical schema if the real extract is absent — see `data/README.md` and run `python scripts/make_synthetic_sample.py`.

In [ ]:
# ---------------------------------------------------------------------------
# Setup — run once. Every later cell depends on this.
# ---------------------------------------------------------------------------
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

from smartnet import config
from smartnet.data import loader, validation
from smartnet.evaluation.splits import summarise_strategies, make_splitter, split_diagnostics
from smartnet.evaluation import metrics as M
from smartnet.models.experiment import run_cv, run_sweep, leakage_table
from smartnet.models.interpret import block_importance, logistic_coefficients
from smartnet.visualization import plots as P

R = config.RESULTS_DIR

# Loaded once and reused throughout.
df = loader.load_analysis_frame()
events = loader.event_summary(df, 'motion_5cat')

print(f'{df.shape[0]:,} labelled epochs x {df.shape[1]} columns')
print(f'{df.event_id.nunique()} motion events across {df.session_date.nunique()} recording days')
print(f'{df.timestamp.min():%Y-%m-%d} to {df.timestamp.max():%Y-%m-%d}')

<a id="part-1"></a>
# Part 1 — Data and study understanding

**Question this notebook answers:** what is in this dataset, what does each variable mean, and what analysis does it make possible?

The scientific context comes from Koudou et al. (2022), *Malaria Journal* 21:85, which established the accelerometer methodology. This extract is a separate, later tagged dataset — the numbers here are not a re-analysis of that paper.

In [ ]:
print(df.shape)
df.head(3)

## Label schemes

Four nested schemes. Coarser schemes collapse classes of finer ones, so they answer progressively easier questions.

In [ ]:
for col, names in config.LABEL_MAPS.items():
    vc = df[col].value_counts().sort_index()
    print(f'--- {col}')
    for k, v in vc.items():
        print(f'   {names[int(k)]:<16} {v:5d}  ({100*v/len(df):5.1f}%)')

The imbalance is the central design constraint: **Enter and Exit are the rarest classes** and the most operationally interesting ones. Any evaluation reported as overall accuracy will be dominated by Nothing and Put up.

## Feature structure

Every row carries a 21-second context window: the tagged second, the 10 preceding seconds, the 10 following seconds, plus aggregates over those windows.

In [ ]:
for block, cols in config.FEATURE_BLOCKS.items():
    print(f'{block:<26} {len(cols):3d} features')
print()
print('Current epoch :', config.CURRENT_EPOCH_FEATURES)
print('Aggregates    :', config.WINDOW_AGGREGATE_FEATURES)

## Temporal structure

This determines the entire experimental design, so it is worth establishing carefully.

In [ ]:
d = df.sort_values('timestamp')
gaps = d['gap_seconds'].dropna()
print('Period      :', df.timestamp.min(), '->', df.timestamp.max())
print('Recording days:', df.session_date.nunique())
print()
print('Gap between consecutive epochs (seconds):')
print(gaps.value_counts().head(6).to_string())
print(f'\n{(gaps==1).sum()} of {len(gaps)} gaps are exactly 1 second')

In [ ]:
events = loader.event_summary(df, 'motion_5cat')
print(f'{len(events)} contiguous events')
print(events.n_epochs.describe().to_string())
events.head()

**Key finding.** Labelled motions are recorded as contiguous runs of 1-second epochs, up to 16 seconds long. Every run is label-pure — no event spans two classes. This gives a legitimate grouping unit for cross-validation, which notebook 05 relies on.

In [ ]:
loader.assert_events_are_label_pure(df)
print('All events label-pure.')

## What this dataset does *not* contain

There is **no participant, device or bed-net identifier**. Consequences:

1. Grouping can be done by event and by recording day, but **not by person**.
2. Performance on a *new participant* — the deployment-relevant question — cannot be estimated.
3. The adult/child subgroup comparison reported in the published study **cannot be reproduced**.

These are stated as limitations throughout rather than worked around.

In [ ]:
missing_ids = [c for c in ['participant_id','device_id','net_id','subject'] if c in df.columns]
print('Identifier columns present:', missing_ids or 'none')

<a id="part-2"></a>
# Part 2 — Data quality

**Question:** is this extract fit for modelling, and what would block it?

Eight automated checks run on every pipeline execution. Each returns a severity rather than raising, so one pass surfaces every problem.

In [ ]:
report = validation.run_all(df)
report

In [ ]:
print('Blocking failure:', validation.has_blocking_failure(report))

## Missingness

In [ ]:
miss = loader.describe_missingness(df)
print(miss.head(10).to_string(index=False))

## Label consistency

The nested schemes must agree. Every row that is *Nothing* in the binary scheme must be *Nothing* in all schemes, and 5-category Enter/Exit must collapse into 4-category *Enter or exit*. This is a real integrity test of the tagging process, not a formality.

In [ ]:
print(pd.crosstab(df.motion_5cat.map(config.LABEL_MAPS['motion_5cat']),
                  df.motion_4cat.map(config.LABEL_MAPS['motion_4cat'])).to_string())

Class support is uneven. The rarest classes have few epochs, so per-class estimates for Enter and Exit carry wide uncertainty under cross-validation. Flagged, not fixed — no amount of resampling creates information that is not there.

In [ ]:
for col in ['motion_4cat','motion_5cat']:
    vc = df[col].value_counts()
    names = config.LABEL_MAPS[col]
    print(f'{col}: rarest = {names[int(vc.idxmin())]} at {vc.min()} epochs '
          f'(~{vc.min()//config.N_SPLITS} per CV fold)')

## Deliberate non-decisions

Three things this pipeline does **not** do, each on purpose:

- **No outlier removal.** Large accelerations are the signal, not noise. Values are range-checked and reported, never clipped.
- **No imputation.** There is nothing missing.
- **No resampling of minority classes.** Class weighting is used inside the models instead, so reported support stays honest.

<a id="part-3"></a>
# Part 3 — Exploratory analysis

**Question:** what does the movement signal actually look like for each behaviour, and is there visible separation before any model is fitted?

In [ ]:
fig = P.plot_class_balance(df, 'motion_5cat'); plt.show()

In [ ]:
fig = P.plot_event_duration(events); plt.show()

Events are short. Most are a handful of seconds, and adjacent epochs within one event share ±10 seconds of context — so they are near-duplicates of each other. Notebook 05 quantifies what that does to a random train/test split.

In [ ]:
fig = P.plot_temporal_coverage(df, 'motion_5cat'); plt.show()

Motions were staged during daytime sessions; *Nothing* was sampled across all 24 hours. This is a property of the data-collection protocol, and it means hour-of-day would be a leaky feature — it is deliberately excluded from the model.

In [ ]:
fig = P.plot_signal_examples(df, 'motion_5cat'); plt.show()

## Feature distributions by class

In [ ]:
names = config.LABEL_MAPS['motion_5cat']
key = ['sum_vectormagnitudes','stdvz','meanzover10vector_back','sumover10vector_forward']
summary = df.groupby(df.motion_5cat.map(names))[key].agg(['mean','std']).round(3)
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, col in zip(axes.ravel(), key):
    for i, (name, sub) in enumerate(df.groupby(df.motion_5cat.map(names))):
        ax.hist(sub[col], bins=30, alpha=0.55, label=name, color=P.PALETTE[i % len(P.PALETTE)])
    ax.set_title(col, fontsize=9); ax.set_yscale('log')
axes[0,0].legend(fontsize=7, frameon=False)
plt.tight_layout(); plt.show()

*Nothing* separates cleanly on every feature — it is near-zero movement. The four motion classes overlap heavily with one another, which previews the result: detecting *that* something happened is easy; identifying *which* motion is not.

## Correlation structure

In [ ]:
corr = df[config.WINDOW_AGGREGATE_FEATURES + config.CURRENT_EPOCH_FEATURES].corr()
fig, ax = plt.subplots(figsize=(6.5,5.2))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=7); ax.grid(False)
fig.colorbar(im, shrink=0.8); plt.title('Aggregate feature correlations'); plt.show()

The lag features are strongly autocorrelated by construction. This is why permutation importance (notebook 07) is read at the level of feature *blocks* rather than trusting individual rankings among correlated columns.

<a id="part-4"></a>
# Part 4 — Feature engineering and selection

**Question:** which parts of the 21-second context window actually carry signal?

The features were engineered by the study team before this extract: per-second summary statistics, ±10-second lags and leads, and aggregates over those windows. Rather than invent more, this notebook tests which existing blocks matter — an ablation, not an expansion.

In [ ]:
for b, c in config.FEATURE_BLOCKS.items():
    print(f'{b:<26} {len(c):3d}')

## Ablation

Each block is evaluated under grouped-event CV with the same random forest. If the 40 individual lag columns add nothing over the 6 aggregates, the feature set can be cut by 85% with no cost — which matters for on-device inference.

In [ ]:
rows = []
for block in config.FEATURE_BLOCKS:
    r = run_cv(df, 'motion_5cat', 'random_forest', 'grouped_event',
               features=config.FEATURE_BLOCKS[block])
    rows.append({'block': block, 'n_features': r['n_features'],
                 'accuracy': round(r['cv_accuracy_mean'], 4),
                 'balanced_accuracy': round(r['cv_balanced_accuracy_mean'], 4),
                 'macro_f1': round(r['cv_macro_f1_mean'], 4)})
ablation = pd.DataFrame(rows).sort_values('balanced_accuracy', ascending=False)
ablation

In [ ]:
fig, ax = plt.subplots(figsize=(7,3.2))
ax.barh(ablation.block, ablation.balanced_accuracy, color='#17365D')
for i,(b,v,n) in enumerate(zip(ablation.block, ablation.balanced_accuracy, ablation.n_features)):
    ax.text(v+0.005, i, f'{v:.3f}  ({n} features)', va='center', fontsize=8)
ax.set_xlim(0,1.05); ax.set_xlabel('Balanced accuracy (grouped-event CV)')
ax.set_title('Feature block ablation'); ax.grid(axis='y', visible=False); plt.show()

## Why no additional features were engineered

Frequency-domain features (FFT, spectral energy) were considered and rejected. They require the raw 10 Hz waveform; this extract contains only per-second summary statistics, so the underlying signal needed to compute them is not present. Claiming spectral features here would mean fabricating them.

Hour-of-day was excluded deliberately: notebook 03 shows motions were staged in daytime sessions, so time of day would separate the classes for reasons that have nothing to do with movement and would not survive deployment.

<a id="part-5"></a>
# Part 5 — Validation design and baseline models

**Question:** how should this dataset be split, and what does a sensible model achieve?

This is the most important notebook in the project.

## The leakage risk

Each labelled motion is a contiguous run of 1-second epochs, and each row carries features for the 10 seconds either side. Adjacent rows within an event overlap in most of their input. Splitting rows at random puts near-duplicates on both sides.

In [ ]:
diag = summarise_strategies(df, 'motion_5cat')
diag[['strategy','grouping','shared_events','pct_test_from_seen_event']]

**Under a random split, a substantial share of test epochs come from a motion event that also appears in training.** Grouped splits reduce this to zero by construction.

In [ ]:
from smartnet.visualization import plots as P
fig = P.plot_leakage_diagnostic(diag); plt.show()

## Does it actually matter?

The diagnostic establishes the *risk*. Whether it inflates scores is an empirical question.

In [ ]:
results, artefacts = run_sweep(df, label_col='motion_5cat')
leakage_table(results)

### Whether it actually matters is an empirical question

The specific outcome of this diagnostic is withheld pending PI review, along with the rest of this analysis's results. The design choice does not depend on that outcome: the structural risk is real and measurable regardless of how much it happens to inflate scores on this particular dataset. **Grouped-event CV is used for every subsequent result.**

In [ ]:
base = results[results.model=='majority_baseline']
print('Majority baseline (grouped-event):')
print(base[base.strategy=='grouped_event'][['cv_accuracy_mean','cv_balanced_accuracy_mean']].round(4).to_string(index=False))
print('\nA model predicting only the most common class reaches 37% accuracy.')
print('Every number below must be read against that floor.')

In [ ]:
fig = P.plot_model_comparison(results); plt.show()

<a id="part-6"></a>
# Part 6 — Model comparison and error analysis

**Question:** which model, which label scheme, and where does it fail?

In [ ]:
res5 = pd.read_csv(R/'results_motion_5cat_all.csv')
res5[res5.strategy=='grouped_event'][
    ['model','cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean','cv_macro_auc_mean']
].round(4)

Results (which model and label scheme perform best) are withheld pending PI review.

Deep learning was rejected, not overlooked: the number of independent events and minority-class examples in this extract is too small to train or validate a 1D CNN or LSTM credibly.

## Accuracy versus balanced accuracy

In [ ]:
fig = P.plot_accuracy_vs_balanced(res5); plt.show()

In [ ]:
cm5 = pd.read_csv(R/'confusion_motion_5cat_grouped_event.csv', index_col=0)
fig = P.plot_confusion(cm5, 'Random forest, grouped-event CV — 5 categories'); plt.show()
pd.read_csv(R/'per_class_motion_5cat_grouped_event.csv').round(3)

## Error analysis: entering versus exiting

In [ ]:
m = cm5.to_numpy()
labels = [c.replace('pred_','') for c in cm5.columns]
i_en, i_ex = labels.index('Enter'), labels.index('Exit')
en_err, ex_err = m[i_en].sum()-m[i_en,i_en], m[i_ex].sum()-m[i_ex,i_ex]
print(f'Enter: {en_err} errors, {m[i_en,i_ex]} of them predicted as Exit ({100*m[i_en,i_ex]/en_err:.0f}%)')
print(f'Exit : {ex_err} errors, {m[i_ex,i_en]} of them predicted as Enter ({100*m[i_ex,i_en]/ex_err:.0f}%)')

Results of the error analysis — which classes are confused with which, and how that compares to the published study — are withheld pending PI review.

## Collapsing the classes

In [ ]:
res4 = pd.read_csv(R/'results_motion_4cat_all.csv')
comp = pd.DataFrame({
 '5-category': res5[(res5.strategy=='grouped_event')&(res5.model=='random_forest')].iloc[0][
     ['cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean']].values,
 '4-category': res4[(res4.strategy=='grouped_event')&(res4.model=='random_forest')].iloc[0][
     ['cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean']].values,
}, index=['accuracy','balanced_accuracy','macro_f1']).astype(float).round(4)
comp

In [ ]:
fig = P.plot_per_class(pd.read_csv(R/'per_class_motion_4cat_grouped_event.csv'),
                       'Per-class performance — 4 categories (grouped-event CV)'); plt.show()

Results of the class-collapsing comparison are withheld pending PI review.

**Recommended approach:** the code below evaluates both the five-category and four-category label schemes; which one is preferable depends on results not published in this notebook.

<a id="part-7"></a>
# Part 7 — Interpretation

**Question:** what is the model relying on, and does it correspond to anything physically sensible?

For a health application, a model nobody can interrogate is hard to govern. Permutation importance is computed on held-out folds under the grouped split — impurity importance from a fitted forest is measured on training data and biased toward high-cardinality features, which is the confound this project is about.

In [ ]:
# Computed here rather than loaded from disk, so the interpretability step is
# reproduced in front of the reader. ~15s: 48 features x 5 repeats x 2 folds.
imp = permutation_importances(df, 'motion_5cat', n_repeats=5, max_folds=2)
imp.head(12).round(4)

In [ ]:
fig = P.plot_feature_importance(imp); plt.show()

In [ ]:
block_importance(imp)

Feature-importance rankings and which blocks dominate are withheld pending PI review. The code below computes permutation importance on held-out folds under the grouped split — impurity importance from a fitted forest is measured on training data and biased toward high-cardinality features, which is the confound this project is about.

## The interpretable model

In [ ]:
coef = logistic_coefficients(df, 'motion_5cat')
top = coef.abs().max(axis=1).sort_values(ascending=False).head(10).index
coef.loc[top].round(3)

## What these numbers are not

Permutation importance measures what a model **relies on**, not what **causes** a behaviour. These are associations between engineered signal features and a human-applied label.

`meanzover10vector_back` being important does not mean z-axis displacement causes someone to enter a net. It means that, in this dataset, with this sensor placement, that statistic helps separate labels a human assigned from video.

**Prediction ≠ causation.** Any operational use must rest on prospective validation, not on feature rankings.

## Deployment implication

The feature set could likely be reduced substantially with little loss — relevant for on-device or low-bandwidth inference in field conditions. The specific reduction achievable is withheld pending PI review.

A calibrated model should also be allowed to **abstain**: output *net crossed* with high confidence, and decline to guess direction if direction is not reliably recoverable. Matching the output to the model's real capability is more useful than forcing a decision it cannot support.

<a id="part-8"></a>
# Part 8 — Summary and limitations

## What this analysis established

Specific results (accuracy, balanced accuracy, sensitivity, feature importance, leakage inflation) are withheld from this notebook pending review and sign-off from the study PI, since they are derived from human-subjects research data. The pipeline, methodology, and code that produce them are public; the numbers are not.

## Limitations

1. **No participant identifier.** Grouping is by event and recording day, not by person, so **performance on a new participant cannot be estimated** — the question that matters most for deployment. This is the single biggest limitation.

2. **No demographics**, so the adult/child fairness comparison reported in the published study cannot be reproduced. Children are a priority malaria population, which makes this a material gap.

3. **Entry/exit direction may not be fully recoverable** from this sensor placement and feature set at these sample sizes — see the interpretation section for reasoning.

4. **Small minority classes.** Enter and Exit have few epochs, so those per-class figures carry wide uncertainty.

5. **Staged conditions.** Motions were recorded in daytime sessions, not during natural overnight net use.

6. **Single sensor placement.** One accelerometer on one side panel.

7. **`Nothing` classification quality deserves scrutiny.** Near-zero movement is trivially separable, but with a participant identifier it would be worth testing whether classification behaviour reflects a recording-context artefact.

Limitations 1 and 2 are blocked by the extract, not by method. **A participant-linked extract from the study PI would be the single biggest upgrade to this work.**

## Next steps

1. Obtain participant identifiers; re-run with participant-level grouping and add the adult/child fairness analysis.
2. Test a second sensor or placement to establish whether entry/exit direction is recoverable at all. This is a data-collection question — no modelling will extract information the signal does not contain.
3. Sequence models over raw 10 Hz signal, once materially more labelled events exist.
4. Reduce to the smallest feature set that carries the signal, for low-power on-device inference.
5. Prospective validation in real household conditions, where any programmatic claim has to be earned.

---

*Research and portfolio project. Not a medical device; not validated for clinical or programmatic use.*